## Importeren van Bibliotheken

In [54]:
import pandas
from sklearn.cluster import KMeans
import sqlite3 as sql
import math
import matplotlib.pyplot as plt
import warnings
import numpy as np
warnings.simplefilter('ignore')
from sklearn.metrics.pairwise import euclidean_distances

## Inlezen van de Data

In [55]:
conn = sql.connect("../Data/HulpmiddelenBS/go_sales_train.sqlite")
print(conn)

## Aanmaken van de Dataframe

In [56]:
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
table_names = pandas.read_sql(tables_query, conn)['name']

def get_dataframe(table_name):
    dataframe = pandas.read_sql(f"SELECT * FROM {table_name}", conn)
    return dataframe

table_names

0           country
1     order_details
2      order_header
3      order_method
4           product
5      product_line
6      product_type
7     retailer_site
8     return_reason
9     returned_item
10     sales_branch
11      sales_staff
Name: name, dtype: object

## Aanmaken van de dataframe & Selecteren van de kolommen

In [57]:
df_sales_branch = get_dataframe('sales_branch')
df_order_header = get_dataframe('order_header')
df_order_details = get_dataframe('order_details')
df_product = get_dataframe('product')


merged_dataframe = pandas.merge(
    df_sales_branch,
    df_order_header[['SALES_BRANCH_CODE', 'ORDER_NUMBER']],
    on='SALES_BRANCH_CODE',
    how='left'
)

merged_dataframe = pandas.merge(
    merged_dataframe,
    df_order_details[['ORDER_NUMBER', 'PRODUCT_NUMBER']],
    on='ORDER_NUMBER',
    how='left'
)

merged_dataframe = pandas.merge(
    merged_dataframe,
    df_product[['PRODUCT_NUMBER', 'PRODUCT_TYPE_CODE']],
    on='PRODUCT_NUMBER',
    how='left'
)

model_dataframe = merged_dataframe.loc[:,['SALES_BRANCH_CODE', 'PRODUCT_TYPE_CODE']]
model_dataframe

,SALES_BRANCH_CODE,PRODUCT_TYPE_CODE
0,6,3
1,6,14
2,6,2
3,6,4
4,6,2
...,...,...
37752,39,9
37753,39,9
37754,39,13
37755,39,14


## One-hot encodig van onafhankelijke niet-numerieke variabelen